# 06 - RQ4: Does a machine-learning forecaster beat a SARIMA baseline for monthly route demand?

* **DV:** monthly passengers on a route (all carriers, scheduled passenger service), T-100 Domestic Segment.
* **Models (all forecast the same held-out final 12 months):**
  1. **Seasonal-naive** benchmark - "same month last year" (a floor any model should beat).
  2. **SARIMA** (0,1,1)(0,1,1)12 on log passengers, fitted per route - the statistical baseline.
  3. **Exponential smoothing (Holt-Winters)** on log passengers, additive damped trend and additive seasonality - a second classical baseline.
  4. **LSTM** (one global network, window = 12 months, recursive 12-step forecast, mean of 3 random seeds) - the deep-learning model.
* **Metrics (identical for every model):** MAE, RMSE and MASE (Hyndman & Athanasopoulos, 2021). Forecast error is an *evaluation metric*, not a second modelled variable.
* **Separated evaluation:** every model is also scored on the *training period* (one-step-ahead fit, months 13 to 43), next to the 12-month recursive test forecast.
* **H0:** mean paired difference in error (LSTM - SARIMA) across routes = 0. **H1:** != 0. Tested with a paired t-test (Wilcoxon as a check).

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from config import *
from results_store import save_result, load_results
from labels import MONTH_NAMES, DOW_NAMES, carrier_name, airport_name
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from metrics import mae, rmse, mase
from plot_style import set_style, save_fig
set_style()
series = pd.read_csv(PROCESSED / "route_monthly_passengers.csv", index_col="period", parse_dates=True)
H = RQ4_TEST_MONTHS
train, test = series.iloc[:-H], series.iloc[-H:]
print(f"{series.shape[1]} routes | {len(series)} months ({series.index.min().date()} to {series.index.max().date()}) | train {len(train)} / test {len(test)} months (minimum routes required: {load_results()['sample_size']['rq4']})")

150 routes | 55 months (2022-01-01 to 2026-07-01) | train 43 / test 12 months (minimum routes required: 88)


## 1. Seasonal-naive and SARIMA (one model per route)

In [2]:
import warnings; warnings.filterwarnings("ignore")                      # SARIMAX prints many start-up notes; they are not errors
snaive = {c: train[c].iloc[-12:].values for c in series.columns}           # forecast = value 12 months earlier
sarima, n_fallback, sarima_in = {}, 0, {}
ets, n_ets_fallback, ets_in = {}, 0, {}
for c in series.columns:
    ylog = np.log(train[c].values)
    try:
        fit = SARIMAX(ylog, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12),
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
        f = np.exp(fit.forecast(H))
        if not np.all(np.isfinite(f)): raise ValueError("non-finite forecast")
        sarima[c] = f; sarima_in[c] = np.exp(np.asarray(fit.fittedvalues))      # one-step-ahead fitted values
    except Exception:
        sarima[c] = snaive[c]; sarima_in[c] = train[c].shift(12).values; n_fallback += 1   # documented fallback, counted below
    try:
        hw = ExponentialSmoothing(ylog, trend="add", damped_trend=True, seasonal="add", seasonal_periods=12).fit()
        f = np.exp(hw.forecast(H))
        if not np.all(np.isfinite(f)): raise ValueError("non-finite forecast")
        ets[c] = f; ets_in[c] = np.exp(np.asarray(hw.fittedvalues))
    except Exception:
        ets[c] = snaive[c]; ets_in[c] = train[c].shift(12).values; n_ets_fallback += 1
print(f"SARIMA fitted for {len(sarima) - n_fallback} routes (fallback {n_fallback}); exponential smoothing fitted for {len(ets) - n_ets_fallback} routes (fallback {n_ets_fallback})")

SARIMA fitted for 150 routes (fallback 0); exponential smoothing fitted for 150 routes (fallback 0)


## 2. LSTM (global model)
Each route is rescaled by its own training mean so one network can learn the shared seasonal shape of all routes.
Training windows are 12 consecutive months -> next month; the last 4 windows of every route are held out for early stopping (time-aware validation, no shuffling across time).

In [3]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
LB, VAL = 12, 4
scale = train.mean()
rel = train / scale

def make_windows(arr):
    Xw = np.array([arr[i:i + LB] for i in range(len(arr) - LB)]); yw = arr[LB:]
    return Xw, yw
Xtr, ytr, Xva, yva = [], [], [], []
for c in series.columns:
    Xw, yw = make_windows(rel[c].values)
    Xtr.append(Xw[:-VAL]); ytr.append(yw[:-VAL]); Xva.append(Xw[-VAL:]); yva.append(yw[-VAL:])
Xtr, ytr, Xva, yva = [np.concatenate(a)[..., None] if a is Xtr or a is Xva else np.concatenate(a) for a in (Xtr, ytr, Xva, yva)]
print("training windows:", Xtr.shape, "validation windows:", Xva.shape)

def fit_lstm(seed):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential([tf.keras.layers.Input((LB, 1)), tf.keras.layers.LSTM(32), tf.keras.layers.Dense(16, activation="relu"), tf.keras.layers.Dense(1)])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    es = tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)
    hist = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=300, batch_size=64, verbose=0, callbacks=[es])
    return m, len(hist.history["loss"])

def insample_lstm(m):
    """One-step-ahead predictions for months 13..T of every route (all training windows), in passengers."""
    out = []
    for c in series.columns:
        Xw, _ = make_windows(rel[c].values)
        out.append(m.predict(Xw[..., None], verbose=0)[:, 0] * scale[c])
    return np.array(out)

def forecast_lstm(m):
    cur = rel.iloc[-LB:].values.T.copy()                                  # routes x 12
    outs = []
    for _ in range(H):
        nxt = m.predict(cur[..., None], verbose=0)[:, 0]
        outs.append(nxt); cur = np.column_stack([cur[:, 1:], nxt])
    return np.array(outs).T * scale.values[:, None]                       # back to passengers

seeds = [42, 43, 44]; lstm_fc, lstm_ins, epochs_used, val_loss = [], [], [], []
for s in seeds:
    m, ep = fit_lstm(s); lstm_fc.append(forecast_lstm(m)); lstm_ins.append(insample_lstm(m)); epochs_used.append(ep)
lstm_mean = np.mean(lstm_fc, axis=0)
lstm = {c: lstm_mean[i] for i, c in enumerate(series.columns)}
lstm_in_mean = np.mean(lstm_ins, axis=0)
lstm_in = {c: lstm_in_mean[i] for i, c in enumerate(series.columns)}
print("epochs used per seed:", epochs_used)

training windows: (4050, 12, 1) validation windows: (600, 12, 1)


epochs used per seed: [71, 40, 43]


## 3. Evaluation on the held-out 12 months

In [4]:
rows = []
for c in series.columns:
    yt, tr = test[c].values, train[c].values
    for name, fc in (("Seasonal naive", snaive[c]), ("Exponential smoothing", ets[c]), ("SARIMA", sarima[c]), ("LSTM", lstm[c])):
        rows.append({"route": c, "model": name, "MAE": mae(yt, fc), "RMSE": rmse(yt, fc), "MASE": mase(yt, fc, tr, 12),
                     "MAPE": float(np.mean(np.abs((yt - fc) / yt)) * 100)})
res = pd.DataFrame(rows); res.to_csv(TAB / "rq4_route_metrics.csv", index=False)
summ = res.groupby("model")[["MAE", "RMSE", "MASE", "MAPE"]].agg(["mean", "median"]).round(3)
summ.to_csv(TAB / "rq4_model_summary.csv")
w = res.pivot(index="route", columns="model", values=["MAE", "RMSE", "MASE"])
# training-period evaluation: one-step-ahead fit, all models scored on the same months. The first 13 months are skipped because
# SARIMA with one regular and one seasonal difference needs 13 observations to start up (its first fitted values are not meaningful).
BURN = 13
trn = []
for c in series.columns:
    ytr = train[c].values; yt_ = ytr[BURN:]
    for name, fit_ in (("Seasonal naive", ytr[BURN - 12:-12]), ("Exponential smoothing", ets_in[c][BURN:]), ("SARIMA", sarima_in[c][BURN:]), ("LSTM", lstm_in[c][BURN - 12:])):
        trn.append({"route": c, "model": name, "MAE": mae(yt_, fit_), "RMSE": rmse(yt_, fit_), "MASE": mae(yt_, fit_) / np.mean(np.abs(ytr[BURN:] - ytr[BURN - 12:-12]))})
train_res = pd.DataFrame(trn); train_res.to_csv(TAB / "rq4_train_metrics.csv", index=False)
train_summ = train_res.groupby("model")[["MAE", "RMSE", "MASE"]].agg(["mean", "median"]).round(3)
train_summ.to_csv(TAB / "rq4_train_summary.csv")
print("training-period (one-step-ahead) fit:"); display(train_summ)
summ

training-period (one-step-ahead) fit:


MAE                RMSE             MASE       
                           mean    median      mean    median   mean median
model                                                                      
Exponential smoothing  2550.136  2479.525  3181.565  3085.762  0.406  0.382
LSTM                   5254.845  5080.066  6435.905  6273.781  0.830  0.783
SARIMA                 4426.055  4214.759  5648.240  5463.653  0.701  0.689
Seasonal naive         6633.081  6359.550  8176.833  7903.870  1.000  1.000

MAE                RMSE             MASE         \
                           mean    median      mean    median   mean median   
model                                                                         
Exponential smoothing  5321.635  4417.379  6261.563  5328.594  0.806  0.697   
LSTM                   7085.207  6553.460  8336.720  7894.028  1.074  0.921   
SARIMA                 5449.081  4704.796  6448.717  5645.237  0.817  0.723   
Seasonal naive         5246.899  4514.333  6159.501  5502.931  0.798  0.714   

                        MAPE         
                        mean median  
model                                
Exponential smoothing  7.385  6.130  
LSTM                   9.911  8.736  
SARIMA                 7.454  6.268  
Seasonal naive         7.251  6.116

In [5]:
def paired(metric):
    d = (w[(metric, "LSTM")] - w[(metric, "SARIMA")]).dropna()
    t, p = stats.ttest_rel(w[(metric, "LSTM")], w[(metric, "SARIMA")])
    try: wp = stats.wilcoxon(d).pvalue
    except Exception: wp = np.nan
    return {"metric": metric, "mean_diff": float(d.mean()), "sd_diff": float(d.std(ddof=1)), "t": float(t), "df": int(len(d) - 1), "p_t": float(p),
            "p_wilcoxon": float(wp), "cohens_dz": float(d.mean() / d.std(ddof=1)), "share_routes_lstm_better": float((d < 0).mean() * 100),
            "ci": [float(d.mean() - 1.96 * d.std(ddof=1) / np.sqrt(len(d))), float(d.mean() + 1.96 * d.std(ddof=1) / np.sqrt(len(d)))]}
tests = {m: paired(m) for m in ("MAE", "RMSE", "MASE")}
pd.DataFrame(tests).T.round(4)

,metric,mean_diff,sd_diff,t,df,p_t,p_wilcoxon,cohens_dz,share_routes_lstm_better,ci
MAE,MAE,1636.1264,3623.83903,5.529598,149,0.0,0.0,0.45149,28.666667,"[1056.191374, 2216.061427]"
RMSE,RMSE,1888.003024,3794.953167,6.09315,149,0.0,0.0,0.497504,29.333333,"[1280.684032, 2495.322016]"
MASE,MASE,0.256955,0.570966,5.511787,149,0.0,0.0,0.450036,28.666667,"[0.165581, 0.348328]"


In [6]:
# --- figures ---------------------------------------------------------------------------------------
tot = series.sum().sort_values(ascending=False)
pick = [tot.index[0], tot.index[len(tot) // 3], tot.index[2 * len(tot) // 3]]
gap = (w[("MAE", "LSTM")] - w[("MAE", "SARIMA")]).abs() / w[("MAE", "SARIMA")]
pick.append(gap.drop(pick, errors="ignore").idxmax())
fig, axs = plt.subplots(2, 2, figsize=(6.5, 4.6), sharex=False)
for ax, c in zip(axs.ravel(), pick):
    ctx = series[c].iloc[-(H + 24):]
    ax.plot(ctx.index, ctx.values / 1e3, color=INK, lw=1.6, label="Actual")
    ax.plot(test.index, snaive[c] / 1e3, color=INK_2, lw=1.3, ls=":", label="Seasonal naive")
    ax.plot(test.index, sarima[c] / 1e3, color=PALETTE[0], lw=1.6, ls="--", label="Seasonal ARIMA")
    ax.plot(test.index, lstm[c] / 1e3, color=PALETTE[1], lw=1.6, label="Long short-term memory")
    ax.set_title(c, fontsize=9, loc="left"); ax.set_ylabel("Passengers (thousands)", fontsize=8)
    ax.tick_params(axis="x", labelrotation=30, labelsize=7)
h_, l_ = axs[0, 0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower center", ncol=4, fontsize=8, frameon=False)
fig.tight_layout(rect=[0, 0.06, 1, 1]); save_fig(fig, "fig19_rq4_forecasts")

fig, ax = plt.subplots(figsize=(6.0, 3.3))
means = w["MASE"].mean(); se = w["MASE"].std() / np.sqrt(len(w))
order = ["Seasonal naive", "Exponential smoothing", "SARIMA", "LSTM"]
full = {"Seasonal naive": "Seasonal naive", "Exponential smoothing": "Exponential\nsmoothing", "SARIMA": "Seasonal autoregressive\nintegrated moving average", "LSTM": "Long short-term\nmemory network"}
ax.bar([full[o] for o in order], [means[o] for o in order], yerr=[1.96 * se[o] for o in order], color=[PALETTE[2], PALETTE[3], PALETTE[0], PALETTE[1]], width=0.6, capsize=3, ecolor=INK_2)
for i, o in enumerate(order): ax.text(i, means[o] + 1.96 * se[o] + 0.02, f"{means[o]:.2f}", ha="center", fontsize=8)
ax.set_ylabel("Mean absolute scaled error across routes\n(lower is better; bars show 95% confidence interval)", fontsize=8)
ax.tick_params(axis="x", labelsize=8)
fig.tight_layout(); save_fig(fig, "fig20_rq4_comparison")

rq4 = {
    "routes": int(series.shape[1]), "min_required": load_results()["sample_size"]["rq4"], "months_total": int(len(series)),
    "train_months": int(len(train)), "test_months": int(len(test)), "train_range": [str(train.index.min().date()), str(train.index.max().date())],
    "test_range": [str(test.index.min().date()), str(test.index.max().date())], "sarima_fallbacks": int(n_fallback),
    "lstm_epochs": epochs_used, "ets_fallbacks": int(n_ets_fallback),
    "train_means": {m: {k: float(train_res[train_res.model == m][k].mean()) for k in ("MAE", "RMSE", "MASE")} for m in order},
    "train_medians": {m: {k: float(train_res[train_res.model == m][k].median()) for k in ("MAE", "RMSE", "MASE")} for m in order}, "summary": res.groupby("model")[["MAE", "RMSE", "MASE", "MAPE"]].agg(["mean", "median"]).round(4).to_json(),
    "means": {m: {k: float(res[res.model == m][k].mean()) for k in ("MAE", "RMSE", "MASE", "MAPE")} for m in order},
    "medians": {m: {k: float(res[res.model == m][k].median()) for k in ("MAE", "RMSE", "MASE", "MAPE")} for m in order},
    "tests": tests,
    "decision_mae": "reject H0" if tests["MAE"]["p_t"] < 0.05 else "fail to reject H0",
    "decision_rmse": "reject H0" if tests["RMSE"]["p_t"] < 0.05 else "fail to reject H0",
    "better_model": "LSTM" if tests["MAE"]["mean_diff"] < 0 else "SARIMA",
    "pick_routes": pick,
}
save_result("rq4", rq4)
print(rq4["decision_mae"], "(MAE) |", rq4["decision_rmse"], "(RMSE)")

reject H0 (MAE) | reject H0 (RMSE)
